# 03 · Análisis de métricas y comparación NLTK vs spaCy

**Etapa 4 del pipeline.**



### Preguntas de investigación elegidas (2 por eje)

| Eje | Pregunta |
|---|---|
| **A1** | ¿Las reseñas de Terror usan más adjetivos que las de Comedia? |
| **A2** | ¿Las reseñas de Acción tienen mayor proporción de verbos que las de Drama? |
| **B1** | ¿Las reseñas con calificación baja usan más negaciones que las de calificación alta? |
| **B6** | ¿Las reseñas negativas son más largas o tienen oraciones más largas que las positivas? |
| **C2** | ¿Ha cambiado la proporción de adjetivos a lo largo del tiempo? |
| **C4** | ¿Cambia con los años la longitud media de las reseñas? |



In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

# Raíz del proyecto: el notebook se ejecuta desde la carpeta notebooks/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import metrics as mt
importlib.reload(mt)

DIR_RESULTADOS = RAIZ / "data" / "results"
RUTA_POS = DIR_RESULTADOS / "pos_analysis.csv"
RUTA_LIMPIAS = RAIZ / "data" / "processed" / "resenas_limpias.csv"
RUTA_METRICAS = DIR_RESULTADOS / "metricas_por_resena.csv"
RUTA_RESPUESTAS = DIR_RESULTADOS / "respuestas_preguntas.csv"
RUTA_REVISION = DIR_RESULTADOS / "revision_manual.csv"

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.float_format", "{:.3f}".format)

# cargar_tokens evita que palabras como "NA" o "null" se lean como valores vacíos
tokens = mt.cargar_tokens(RUTA_POS)
resenas = pd.read_csv(RUTA_LIMPIAS)
resenas["valoracion"] = pd.Categorical(resenas["valoracion"], categories=["negativa", "intermedia", "positiva"], ordered=True)

print(f"{len(tokens):,} tokens de {tokens['id'].nunique()} reseñas")

3,494,019 tokens de 16991 reseñas


## 1. Métricas por reseña


| Métrica | Cálculo |
|---|---|
| Densidad léxica | (NOUN + VERB + ADJ + ADV) / tokens, en % |
| Ratio verbos/sustantivos | VERB / NOUN (alto = narrativo, bajo = descriptivo) |
| Índice de verbos y de adjetivación | VERB y ADJ por cada 100 tokens |
| Índice de negación | negaciones por cada 100 tokens |
| Pronombres por persona | PRON con `Person=1`, `2` o `3` por cada 100 tokens |
| Diversidad léxica (TTR) | lemas distintos / 50, sobre los **primeros 50 tokens** de cada reseña |
| Longitud media de oración | tokens / oraciones (segmentación de spaCy) |


In [2]:
# se hace con 50 (longitud de texto), porque asi quedan menos por fuera (51)
metricas = mt.metricas_por_resena(tokens, tamano_ttr=50)
df = resenas[["id", "genero", "periodo", "anio", "calificacion", "valoracion", "n_palabras"]].merge(metricas, on="id")

df.to_csv(RUTA_METRICAS, index=False, encoding="utf-8")
print(f"Guardado {RUTA_METRICAS.relative_to(RAIZ)}: {len(df)} reseñas")
print(f"Reseñas sin TTR (menos de 50 tokens): {df['ttr'].isna().sum()}")
df[list(mt.NOMBRES_METRICAS)].describe().T.rename(index=mt.NOMBRES_METRICAS)

Guardado data\results\metricas_por_resena.csv: 16991 reseñas
Reseñas sin TTR (menos de 50 tokens): 957


,count,mean,std,min,25%,50%,75%,max
Tokens (sin puntuación),16991.000,179.016,92.539,27.000,118.000,159.000,235.000,457.000
Densidad léxica (%),16991.000,45.078,4.303,0.000,42.446,45.087,47.647,70.968
Ratio verbos/sustantivos,16988.000,0.592,0.268,0.000,0.429,0.556,0.707,9.000
Verbos por 100 tokens,16991.000,10.520,3.065,0.000,8.497,10.370,12.389,35.714
Adjetivos por 100 tokens,16991.000,8.751,2.998,0.000,6.763,8.491,10.377,40.081
Negaciones por 100 tokens,16991.000,1.993,1.640,0.000,0.855,1.724,2.766,33.333
Pronombres 1.ª persona por 100 tokens,16991.000,0.867,1.140,0.000,0.000,0.585,1.311,33.333
Pronombres 2.ª persona por 100 tokens,16991.000,0.332,0.685,0.000,0.000,0.000,0.478,10.000
Pronombres 3.ª persona por 100 tokens,16991.000,2.323,1.485,0.000,1.351,2.174,3.125,16.129
"Diversidad léxica (TTR, primeros 50 tokens)",16034.000,0.746,0.063,0.060,0.700,0.740,0.780,1.000


### 1.1 Promedios por grupo

Panorama de todas las métricas por género, valoración y periodo. Las preguntas de la sección 2 profundizan en algunas de ellas.

In [3]:
mt.resumen_por_grupo(df, "genero").rename(columns=mt.NOMBRES_METRICAS)

,n,Tokens (sin puntuación),Densidad léxica (%),Ratio verbos/sustantivos,Verbos por 100 tokens,Adjetivos por 100 tokens,Negaciones por 100 tokens,Pronombres 1.ª persona por 100 tokens,Pronombres 2.ª persona por 100 tokens,Pronombres 3.ª persona por 100 tokens,"Diversidad léxica (TTR, primeros 50 tokens)",Longitud media de oración (tokens)
genero,,,,,,,,,,,,
Action,3400,180.469,44.684,0.587,10.311,8.667,2.023,0.805,0.310,2.250,0.747,20.484
Animation,3394,182.427,44.943,0.576,10.224,8.993,1.816,0.932,0.299,2.209,0.743,21.115
Comedy,3399,175.024,44.669,0.600,10.480,8.824,1.985,0.842,0.333,2.394,0.750,20.472
Drama,3400,179.762,45.042,0.591,10.608,8.672,1.939,0.874,0.331,2.403,0.746,21.183
Horror,3398,177.400,46.050,0.603,10.978,8.601,2.204,0.882,0.387,2.360,0.745,21.162


In [4]:
mt.resumen_por_grupo(df, "valoracion").rename(columns=mt.NOMBRES_METRICAS)

,n,Tokens (sin puntuación),Densidad léxica (%),Ratio verbos/sustantivos,Verbos por 100 tokens,Adjetivos por 100 tokens,Negaciones por 100 tokens,Pronombres 1.ª persona por 100 tokens,Pronombres 2.ª persona por 100 tokens,Pronombres 3.ª persona por 100 tokens,"Diversidad léxica (TTR, primeros 50 tokens)",Longitud media de oración (tokens)
valoracion,,,,,,,,,,,,
negativa,3748,172.723,45.994,0.617,11.171,8.423,2.697,0.839,0.268,2.381,0.749,19.823
intermedia,2296,180.787,45.338,0.583,10.318,8.797,2.224,0.790,0.244,2.426,0.751,21.383
positiva,10947,180.799,44.709,0.585,10.339,8.854,1.704,0.893,0.373,2.282,0.744,21.141


In [5]:
mt.resumen_por_grupo(df, "periodo").rename(columns=mt.NOMBRES_METRICAS)

,n,Tokens (sin puntuación),Densidad léxica (%),Ratio verbos/sustantivos,Verbos por 100 tokens,Adjetivos por 100 tokens,Negaciones por 100 tokens,Pronombres 1.ª persona por 100 tokens,Pronombres 2.ª persona por 100 tokens,Pronombres 3.ª persona por 100 tokens,"Diversidad léxica (TTR, primeros 50 tokens)",Longitud media de oración (tokens)
periodo,,,,,,,,,,,,
1998-2009,7499,178.987,44.695,0.596,10.484,8.611,1.969,0.820,0.325,2.236,0.746,19.503
2010-2018,9492,179.039,45.380,0.588,10.549,8.862,2.012,0.905,0.338,2.392,0.746,21.973


## 2. Preguntas de investigación



### Eje A · Comparación entre géneros

#### A1. ¿Las reseñas de Terror usan más adjetivos que las de Comedia?
Métrica: **adjetivos por cada 100 tokens**

In [6]:
display(mt.mostrar(mt.comparar_grupos(df, "idx_adjetivos", "genero", "Horror", "Comedy")))
df.groupby("genero")["idx_adjetivos"].agg(["mean", "median", "std", "size"]).sort_values("mean", ascending=False)

,métrica,grupo A,media A,mediana A,n A,grupo B,media B,mediana B,n B,diferencia (A − B),diferencia (%)
0,Adjetivos por 100 tokens,Horror,8.601,8.333,3398,Comedy,8.824,8.586,3399,-0.222,-2.520


,mean,median,std,size
genero,,,,
Animation,8.993,8.719,3.053,3394
Comedy,8.824,8.586,3.068,3399
Drama,8.672,8.427,3.000,3400
Action,8.667,8.403,2.936,3400
Horror,8.601,8.333,2.914,3398


**Interpretación A1.** **No.** Las reseñas de Terror no usan más adjetivos que las de Comedia. Tienen en promedio 8,25 adjetivos por cada 100 tokens, frente a 8,58 en Comedia: una diferencia de apenas 0,3 adjetivos (4 %), y además a favor de Comedia. Terror es incluso el género con el índice de adjetivación más bajo de los cinco, aunque las diferencias entre géneros son de décimas sobre una media cercana a 8,5.


#### A2. ¿Las reseñas de Acción tienen mayor proporción de verbos que las de Drama?
Métrica principal: **verbos por cada 100 tokens**. Se muestra también el **ratio verbos/sustantivos**

In [7]:
display(mt.mostrar(mt.comparar_grupos(df, "idx_verbos", "genero", "Action", "Drama")))
df.groupby("genero")[["idx_verbos", "ratio_verbos_sustantivos"]].mean().sort_values("idx_verbos", ascending=False)

,métrica,grupo A,media A,mediana A,n A,grupo B,media B,mediana B,n B,diferencia (A − B),diferencia (%)
0,Verbos por 100 tokens,Action,10.311,10.149,3400,Drama,10.608,10.435,3400,-0.297,-2.804


,idx_verbos,ratio_verbos_sustantivos
genero,,
Horror,10.978,0.603
Drama,10.608,0.591
Comedy,10.480,0.600
Action,10.311,0.587
Animation,10.224,0.576


**Interpretación A2.** **No; ocurre lo contrario.** Las reseñas de Acción tienen *menos* verbos que las de Drama: 10,13 frente a 10,73 por cada 100 tokens. Acción es el género con menos verbos y el de ratio verbos/sustantivos más bajo, y Terror el de más verbos. La hipótesis "más acción en pantalla → más verbos en la reseña" no se cumple: las reseñas de Acción parecen más descriptivas (efectos, escenas, actores) que narrativas.


### Eje B · Emocionalidad gramatical

#### B1. ¿Las reseñas con calificación baja usan más negaciones que las de calificación alta?
Métrica: **negaciones por cada 100 tokens**. Se compara la valoración negativa (1–4) con la positiva (7–10), y se revisa toda la escala de 1 a 10.

In [8]:
display(mt.mostrar(mt.comparar_grupos(df, "idx_negacion", "valoracion", "negativa", "positiva")))
df.groupby("calificacion")["idx_negacion"].agg(["mean", "median", "size"]).T

,métrica,grupo A,media A,mediana A,n A,grupo B,media B,mediana B,n B,diferencia (A − B),diferencia (%)
0,Negaciones por 100 tokens,negativa,2.697,2.381,3748,positiva,1.704,1.493,10947,0.993,58.240


calificacion,1,2,3,4,5,6,7,8,9,10
mean,2.825,2.645,2.601,2.542,2.373,2.114,1.817,1.662,1.609,1.727
median,2.500,2.273,2.350,2.190,2.105,1.813,1.600,1.471,1.408,1.515
size,1687.000,606.000,682.000,773.000,974.000,1322.000,1981.000,2532.000,2257.000,4177.000


**Interpretación B1.** **Sí, y es el resultado más claro de todo el análisis.** Las reseñas negativas usan unas 2,6 negaciones por cada 100 tokens, frente a 1,7 en las positivas

quien critica una obra rechaza o contradice ("no funciona", "nunca llega", "nada original").

#### B6. ¿Las reseñas negativas son más largas o tienen oraciones más largas que las positivas?
Métricas: **longitud de la reseña** (tokens sin puntuación) y **longitud media de oración** (tokens por oración).

In [9]:
mt.mostrar([mt.comparar_grupos(df, "n_tokens", "valoracion", "negativa", "positiva"),
         mt.comparar_grupos(df, "long_oracion", "valoracion", "negativa", "positiva")])

,métrica,grupo A,media A,mediana A,n A,grupo B,media B,mediana B,n B,diferencia (A − B),diferencia (%)
0,Tokens (sin puntuación),negativa,172.723,153.000,3748,positiva,180.799,160.000,10947,-8.076,-4.467
1,Longitud media de oración (tokens),negativa,19.823,17.300,3748,positiva,21.141,18.444,10947,-1.318,-6.234


In [10]:
df.groupby("valoracion", observed=True)[["n_tokens", "long_oracion"]].agg(["mean", "median"])

n_tokens         long_oracion       
               mean  median         mean median
valoracion                                     
negativa    172.723 153.000       19.823 17.300
intermedia  180.787 164.000       21.383 18.845
positiva    180.799 160.000       21.141 18.444

**Interpretación B6.** **No.** Las reseñas negativas y las positivas tienen prácticamente la misma longitud (unos 182 frente a 184 tokens de media, con la misma mediana de 161) y oraciones de longitud equivalente (19,9 frente a 20,3 tokens).

### Eje C · Evolución temporal

#### C2. ¿Ha cambiado la proporción de adjetivos a lo largo del tiempo?
Métrica: **adjetivos por cada 100 tokens**.

In [11]:
mt.mostrar([mt.comparar_grupos(df, "idx_adjetivos", "periodo", "2010-2018", "1998-2009")])

,métrica,grupo A,media A,mediana A,n A,grupo B,media B,mediana B,n B,diferencia (A − B),diferencia (%)
0,Adjetivos por 100 tokens,2010-2018,8.862,8.559,9492,1998-2009,8.611,8.411,7499,0.251,2.916


In [12]:
df.groupby("anio")["idx_adjetivos"].agg(["mean", "size"]).T

anio,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
mean,8.810,8.883,8.999,9.075,8.924,8.812,8.626,8.348,8.264,8.626,...,8.940,9.835,9.017,10.196,9.728,10.728,10.881,9.842,10.229,9.647
size,117.000,465.000,488.000,554.000,545.000,611.000,513.000,849.000,1117.000,788.000,...,1504.000,273.000,231.000,164.000,118.000,85.000,33.000,35.000,19.000,15.000


**Interpretación C2.** **No.** La proporción de adjetivos es estable: 8,50 por cada 100 tokens en 1998–2009 y 8,58 en 2010–2018.

#### C4. ¿Cambia con los años la longitud media de las reseñas?
Métrica: **tokens por reseña** (sin puntuación). Como comprobación, se repite con el número de palabras del **texto original en inglés** (`n_palabras`), para descartar que el cambio lo haya introducido la traducción.

In [13]:
mt.mostrar([mt.comparar_grupos(df, "n_tokens", "periodo", "2010-2018", "1998-2009"),
         mt.comparar_grupos(df, "n_palabras", "periodo", "2010-2018", "1998-2009")])

,métrica,grupo A,media A,mediana A,n A,grupo B,media B,mediana B,n B,diferencia (A − B),diferencia (%)
0,Tokens (sin puntuación),2010-2018,179.039,160.000,9492,1998-2009,178.987,157.000,7499,0.052,0.029
1,n_palabras,2010-2018,173.296,155.500,9492,1998-2009,172.928,153.000,7499,0.368,0.213


In [14]:
df.groupby("anio")["n_tokens"].agg(["mean", "median", "size"]).T

anio,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
mean,122.419,134.178,139.764,143.599,141.787,148.566,186.140,194.655,196.150,206.405,...,159.671,102.267,91.437,82.866,132.644,113.906,100.030,86.543,125.632,164.600
median,102.000,110.000,115.000,119.000,119.000,122.000,157.000,168.000,173.000,176.000,...,135.500,70.000,65.000,67.500,121.500,82.000,85.000,59.000,96.000,182.000
size,117.000,465.000,488.000,554.000,545.000,611.000,513.000,849.000,1117.000,788.000,...,1504.000,273.000,231.000,164.000,118.000,85.000,33.000,35.000,19.000,15.000


**Interpretación C4.** **Sí.** Las reseñas recientes son más largas: 196,5 tokens de media en 2010–2018 frente a 173,9 en 1998–2009, unos 23 tokens más (+13 %)

## 3. Comparación NLTK vs spaCy

Ambas herramientas etiquetaron **los mismos tokens** (la tokenización de spaCy), así que la comparación es directa, token a token

### 3.1 Concordancia global

In [15]:
print(f"Concordancia global NLTK–spaCy (sin puntuación): {mt.concordancia_global(tokens):.1f} %")

Concordancia global NLTK–spaCy (sin puntuación): 82.8 %


### 3.2 Concordancia por categoría



In [16]:
mt.concordancia_por_categoria(tokens)

,tokens spaCy,% coincide (según spaCy),tokens NLTK,% coincide (según NLTK)
NOUN,572128,95.270,897013,60.764
DET,434148,98.124,477348,89.244
ADP,410775,99.870,411769,99.629
VERB,317037,70.376,276987,80.552
ADJ,260018,66.250,190530,90.412
PRON,218668,80.279,200663,87.482
ADV,211090,92.915,204502,95.908
PROPN,181531,8.367,20306,74.796
AUX,179709,71.872,130295,99.129
CCONJ,125873,99.024,124941,99.762


**Interpretación:**
- **Coinciden casi siempre** en las categorías preposiciones (ADP), conjunciones (CCONJ), determinantes (DET) y adverbios (ADV), todas por encima del 93 % según spaCy.
- **Coinciden menos** en las categorías abiertas (VERB, ADJ) y en AUX, y **muy poco en PROPN**. NLTK casi nunca etiqueta nombres propios
- NLTK etiqueta como NOUN muchos más tokens que spaCy. Es el efecto del respaldo `DefaultTagger("NOUN")`: toda palabra desconocida acaba siendo sustantivo

### 3.3 Matriz de confusión

Filas = etiqueta de spaCy; columnas = etiqueta de NLTK. Cada fila suma 100 %: muestra cómo etiquetó NLTK los tokens de cada categoría de spaCy

In [17]:
matriz = mt.matriz_confusion(tokens)
matriz.style.background_gradient(cmap="Blues", axis=None).format("{:.1f}")

upos_nltk,ADJ,ADP,ADV,AUX,CCONJ,DET,INTJ,NOUN,NUM,PRON,PROPN,PUNCT,SCONJ,VERB,X
upos_spacy,,,,,,,,,,,,,,,
ADJ,66.3,0.0,0.5,0.0,0.0,0.7,0.1,30.3,0.0,0.0,0.2,0.0,0.0,1.9,0.0
ADP,0.0,99.9,0.0,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ADV,0.5,0.0,92.9,0.1,0.1,0.2,0.0,4.2,0.0,1.5,0.2,0.0,0.1,0.2,0.0
AUX,0.0,0.0,0.2,71.9,0.0,0.0,0.0,3.9,0.0,0.0,0.2,0.0,0.0,23.8,0.0
CCONJ,0.0,0.0,0.0,0.0,99.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.8,0.0,0.0
DET,0.0,0.3,0.2,0.0,0.0,98.1,0.0,0.1,0.0,1.3,0.0,0.0,0.0,0.0,0.0
INTJ,19.5,0.1,32.6,0.3,0.0,0.2,17.0,24.1,0.0,0.7,2.5,0.0,0.0,3.1,0.0
NOUN,1.8,0.0,0.4,0.1,0.0,0.5,0.0,95.3,0.4,0.1,0.6,0.0,0.0,0.8,0.0
NUM,0.1,0.0,0.1,0.0,0.0,29.5,0.0,14.3,51.2,4.8,0.0,0.1,0.0,0.0,0.0


**Interpretación de la matriz:**
- **Columna NOUN:** es el principal destino de las discrepancias. Allí van la mayoría de los PROPN de spaCy y cerca de un 30 % de sus VERB y ADJ. Son palabras que NLTK no conoce y a las que asigna la etiqueta por defecto.
- **AUX → VERB** (24 % de los AUX de spaCy): no es un error, sino un **criterio distinto**. spaCy trata *estar*, *poder*, *ir* o *deber* como auxiliares; `cess_esp` los anota como verbos principales.
- **PRON → DET** (18 %): palabras como *esta*, *una*, *todo* o *lo*, que pueden ser determinante ("esta película") o pronombre ("esta es buena"). Es **ambigüedad real**: el bigrama de NLTK no siempre la resuelve.


### 3.6 Revisión manual de 50 tokens en discrepancia


In [18]:
disc = mt.discrepancias(tokens)
if RUTA_REVISION.exists():
    print(f"{RUTA_REVISION.relative_to(RAIZ)} ya existe: no se sobrescribe.")
else:
    muestra = disc.sample(50, random_state=42).merge(resenas[["id", "texto_en"]], on="id")
    plantilla = muestra[["id", "oracion", "token", "lema", "upos_spacy", "upos_nltk", "en_cess",
                         "causa_probable", "contexto", "texto_en"]].copy()
    plantilla["correcta"] = ""
    plantilla["causa_revisada"] = ""
    plantilla["comentario"] = ""
    plantilla.to_csv(RUTA_REVISION, index=False, encoding="utf-8-sig")  # utf-8-sig: Excel muestra bien las tildes
    print(f"Creada {RUTA_REVISION.relative_to(RAIZ)} con {len(plantilla)} tokens para revisar.")

data\results\revision_manual.csv ya existe: no se sobrescribe.


### 3.7 Resultado de la revisión manual

In [19]:
revision = pd.read_csv(RUTA_REVISION, keep_default_na=False)
completas = revision[revision["correcta"].str.strip() != ""]

if len(completas) < len(revision):
    print(f"Revisión incompleta: {len(completas)} de {len(revision)} tokens revisados. "
          "Hay que completar las columnas 'correcta' y 'causa_revisada' y volver a ejecutar esta celda.")
else:
    print("¿Quién acertó? (% de los 50 tokens)")
    display((completas["correcta"].str.strip().str.lower().value_counts(normalize=True) * 100).round(1).to_frame("%"))
    print("Causa según la revisión manual:")
    display(completas["causa_revisada"].str.strip().str.lower().value_counts().to_frame("tokens"))
    print("Pre-clasificación automática vs revisión manual:")
    display(pd.crosstab(completas["causa_probable"], completas["causa_revisada"].str.strip().str.lower()))

¿Quién acertó? (% de los 50 tokens)


,%
correcta,
spacy,88.000
nltk,12.000


Causa según la revisión manual:


,tokens
causa_revisada,
ausente,28
ambigüedad,15
criterio,6
traducción,1


Pre-clasificación automática vs revisión manual:


causa_revisada,ambigüedad,ausente,criterio,traducción
causa_probable,,,,
Ambigüedad real,9,0,0,0
Diferencia de criterio entre esquemas,6,0,6,0
Palabra ausente del entrenamiento de NLTK,0,28,0,1


### 3.8 Conclusión: ¿qué herramienta es más confiable para este corpus?

Las dos herramientas coinciden en cerca del 83 % de los tokens. El análisis de las discrepancias indica que **spaCy es la herramienta más confiable para este corpus**: